# Notebook 09 Part 1B: Reading and Writing Parquet in PySpark

## Lecture 15B: Parquet File Handling in PySpark — Practical Data Lake Format

This notebook belongs to:

```text
Phase 2 — Data Wrangling & Spark SQL
```

Parquet is one of the most important file formats in modern data engineering.

It is commonly used in:

```text
Data lakes
ETL pipelines
Analytics tables
Big data storage
Cloud storage systems
```

## 1. Windows Local Setup

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
os.environ["HADOOP_HOME"] = "C:/hadoop"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin;" + os.environ["PATH"]

## 2. Create SparkSession

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Parquet_File_Handling") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

## 3. Create Demo Folder

In [3]:
from pathlib import Path

base_path = Path("pyspark_parquet_demo")
base_path.mkdir(exist_ok=True)

print("Demo folder:", base_path.resolve())

Demo folder: E:\data\Documents\Data_Engineering\Module4-Pyspark\pyspark_parquet_demo


## 4. What Is Parquet?

Parquet is a:

```text
Columnar
Binary
Schema-aware
Compressed
Big-data file format
```

Simple meaning:

```text
Parquet stores data column-wise and preserves schema.
```

## 5. Row-Oriented vs Columnar Storage

Suppose we have this table:

| customer | city | amount |
|---|---|---:|
| Amit | Delhi | 5000 |
| Priya | Mumbai | 7000 |

A row-oriented format stores data conceptually like:

```text
Amit, Delhi, 5000
Priya, Mumbai, 7000
```

A columnar format stores data conceptually like:

```text
customer → Amit, Priya
city     → Delhi, Mumbai
amount   → 5000, 7000
```

Columnar storage helps when we read only a few columns from a large dataset.

## 6. Create Sample Sales DataFrame

In [4]:
sales_data = [
    (101, 1, "Amit", "Delhi", "Laptop", 1, 55000.0),
    (102, 2, "Priya", "Mumbai", "Mobile", 2, 40000.0),
    (103, 3, "Rahul", "Delhi", "Tablet", 1, 25000.0),
    (104, 4, "Sneha", "Pune", "Laptop", 1, 60000.0),
    (105, 5, "Karan", "Mumbai", "Monitor", 2, 30000.0),
    (106, 6, "Meera", "Delhi", "Mobile", 1, 22000.0)
]

sales_columns = [
    "order_id",
    "customer_id",
    "customer_name",
    "city",
    "product",
    "quantity",
    "amount"
]

sales_df = spark.createDataFrame(
    sales_data,
    sales_columns
)

sales_df.show()
sales_df.printSchema()

+--------+-----------+-------------+------+-------+--------+-------+
|order_id|customer_id|customer_name|  city|product|quantity| amount|
+--------+-----------+-------------+------+-------+--------+-------+
|     101|          1|         Amit| Delhi| Laptop|       1|55000.0|
|     102|          2|        Priya|Mumbai| Mobile|       2|40000.0|
|     103|          3|        Rahul| Delhi| Tablet|       1|25000.0|
|     104|          4|        Sneha|  Pune| Laptop|       1|60000.0|
|     105|          5|        Karan|Mumbai|Monitor|       2|30000.0|
|     106|          6|        Meera| Delhi| Mobile|       1|22000.0|
+--------+-----------+-------------+------+-------+--------+-------+

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- amount: double (nullable = true)



## 7. Write DataFrame as Parquet

In [5]:
parquet_path = base_path / "sales_parquet"


sales_df.write \
    .mode("overwrite") \
    .parquet(str(parquet_path))

print(parquet_path.resolve())

E:\data\Documents\Data_Engineering\Module4-Pyspark\pyspark_parquet_demo\sales_parquet


**It will store schema and column names**
<br> **Parquet will be stored as folder**

## 8. Inspect Parquet Output Folder

In [8]:
for file in parquet_path.iterdir():
    print(file.name)

.part-00000-476d25bf-716c-44cf-876a-1cbf7daff8f4-c000.snappy.parquet.crc
._SUCCESS.crc
part-00000-476d25bf-716c-44cf-876a-1cbf7daff8f4-c000.snappy.parquet
_SUCCESS


Spark writes a folder containing files like:

```text
part-00000-...
_SUCCESS
```

Actual data is inside `part-*` files.

`_SUCCESS` only confirms successful write.

## 9. Read Parquet

In [6]:
spark.read.parquet(str(parquet_path))

DataFrame[order_id: bigint, customer_id: bigint, customer_name: string, city: string, product: string, quantity: bigint, amount: double]

In [7]:
sales_parquet_df = spark.read.parquet(str(parquet_path))

sales_parquet_df.show()
sales_parquet_df.printSchema()

+--------+-----------+-------------+------+-------+--------+-------+
|order_id|customer_id|customer_name|  city|product|quantity| amount|
+--------+-----------+-------------+------+-------+--------+-------+
|     101|          1|         Amit| Delhi| Laptop|       1|55000.0|
|     102|          2|        Priya|Mumbai| Mobile|       2|40000.0|
|     103|          3|        Rahul| Delhi| Tablet|       1|25000.0|
|     104|          4|        Sneha|  Pune| Laptop|       1|60000.0|
|     105|          5|        Karan|Mumbai|Monitor|       2|30000.0|
|     106|          6|        Meera| Delhi| Mobile|       1|22000.0|
+--------+-----------+-------------+------+-------+--------+-------+

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- amount: double (nullable = true)



Notice:

```text
No header=True required.
No inferSchema=True required.
```

Parquet stores schema with the data.

## 10. Write Modes with Parquet

| Mode | Meaning |
|---|---|
| `overwrite` | Replace existing data |
| `append` | Add new data files |
| `ignore` | Do nothing if path already exists |
| `error` | Throw error if path exists |

### Overwrite Mode

In [8]:
sales_df.show()

+--------+-----------+-------------+------+-------+--------+-------+
|order_id|customer_id|customer_name|  city|product|quantity| amount|
+--------+-----------+-------------+------+-------+--------+-------+
|     101|          1|         Amit| Delhi| Laptop|       1|55000.0|
|     102|          2|        Priya|Mumbai| Mobile|       2|40000.0|
|     103|          3|        Rahul| Delhi| Tablet|       1|25000.0|
|     104|          4|        Sneha|  Pune| Laptop|       1|60000.0|
|     105|          5|        Karan|Mumbai|Monitor|       2|30000.0|
|     106|          6|        Meera| Delhi| Mobile|       1|22000.0|
+--------+-----------+-------------+------+-------+--------+-------+



In [9]:
mode_demo_path = base_path / "mode_demo_parquet"

sales_df.limit(3).write \
    .mode("overwrite") \
    .parquet(str(mode_demo_path))

print("Rows after overwrite:", spark.read.parquet(str(mode_demo_path)).count())

Rows after overwrite: 3


### Append Mode

In [10]:
sales_df.filter(sales_df.customer_id > 3).write \
    .mode("append") \
    .parquet(str(mode_demo_path))

print("Rows after append:", spark.read.parquet(str(mode_demo_path)).count())

Rows after append: 6


### Ignore Mode

In [11]:
sales_df.write \
    .mode("ignore") \
    .parquet(str(mode_demo_path))

print("Existing path was ignored. Row count:",
      spark.read.parquet(str(mode_demo_path)).count())

Existing path was ignored. Row count: 6


### Error Mode

`error` is the default behavior.

This will fail if path already exists:

```python
sales_df.write.mode("error").parquet(path)
```

We will not run it here because the path already exists.

## 11. Parquet Compression

In [12]:
snappy_path = base_path / "sales_parquet_snappy"

sales_df.write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(str(snappy_path))

for file in snappy_path.iterdir():
    print(file.name)

.part-00000-400d0e8e-7d7c-4aa5-a0ae-6c21a17e12ed-c000.snappy.parquet.crc
._SUCCESS.crc
part-00000-400d0e8e-7d7c-4aa5-a0ae-6c21a17e12ed-c000.snappy.parquet
_SUCCESS


Common compression codecs:

| Codec | Simple meaning |
|---|---|
| `snappy` | Fast and commonly used |
| `gzip` | More compression but slower |
| `uncompressed` | No compression |

## 12. Column Pruning

Parquet is columnar. If we read only selected columns, Spark can avoid reading unnecessary columns.

This is called:

```text
Column Pruning
```

In [18]:
sales_df.select(
    "city",
    "amount"
).show()

+------+-------+
|  city| amount|
+------+-------+
| Delhi|55000.0|
|Mumbai|40000.0|
| Delhi|25000.0|
|  Pune|60000.0|
|Mumbai|30000.0|
| Delhi|22000.0|
+------+-------+



## 15. Write Partitioned Parquet

In [14]:
sales_df.show()

+--------+-----------+-------------+------+-------+--------+-------+
|order_id|customer_id|customer_name|  city|product|quantity| amount|
+--------+-----------+-------------+------+-------+--------+-------+
|     101|          1|         Amit| Delhi| Laptop|       1|55000.0|
|     102|          2|        Priya|Mumbai| Mobile|       2|40000.0|
|     103|          3|        Rahul| Delhi| Tablet|       1|25000.0|
|     104|          4|        Sneha|  Pune| Laptop|       1|60000.0|
|     105|          5|        Karan|Mumbai|Monitor|       2|30000.0|
|     106|          6|        Meera| Delhi| Mobile|       1|22000.0|
+--------+-----------+-------------+------+-------+--------+-------+



In [13]:
partitioned_path = base_path / "sales_by_city"

sales_df.write \
    .mode("overwrite") \
    .partitionBy("city") \
    .parquet(str(partitioned_path))

for item in sorted(partitioned_path.iterdir()):
    print(item.name)

._SUCCESS.crc
_SUCCESS
city=Delhi
city=Mumbai
city=Pune


Folder structure may look like:

```text
sales_by_city/
    city=Delhi/
    city=Mumbai/
    city=Pune/
```

This is file partitioning or directory partitioning.

## 16. Read Partitioned Parquet

In [15]:
partitioned_df = spark.read.parquet(str(partitioned_path))

partitioned_df.show()
partitioned_df.printSchema()

+--------+-----------+-------------+-------+--------+-------+------+
|order_id|customer_id|customer_name|product|quantity| amount|  city|
+--------+-----------+-------------+-------+--------+-------+------+
|     101|          1|         Amit| Laptop|       1|55000.0| Delhi|
|     103|          3|        Rahul| Tablet|       1|25000.0| Delhi|
|     106|          6|        Meera| Mobile|       1|22000.0| Delhi|
|     102|          2|        Priya| Mobile|       2|40000.0|Mumbai|
|     105|          5|        Karan|Monitor|       2|30000.0|Mumbai|
|     104|          4|        Sneha| Laptop|       1|60000.0|  Pune|
+--------+-----------+-------------+-------+--------+-------+------+

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- amount: double (nullable = true)
 |-- city: string (nullable = true)



## 17. Partition Pruning

In [ ]:
partitioned_df.filter(
    partitioned_df.city == "Delhi"
).explain()

When filtering on a partition column, Spark can often skip unrelated folders.

This is called:

```text
Partition Pruning
```

## 18. Spark Partition vs File Partition

| Concept | Meaning | Example |
|---|---|---|
| Spark partition | In-memory chunk processed by task | `df.repartition(4)` |
| File partition | Folder organization by column | `df.write.partitionBy("city")` |

Simple rule:

```text
repartition() affects processing parallelism.
partitionBy() affects output folder layout.
```

## 19. Check Number of Spark Partitions

In [ ]:
print("Partitions:", sales_df.rdd.getNumPartitions())

## 20. Use `repartition()` Before Writing

In [16]:
repartitioned_path = base_path / "sales_repartition_4"

sales_df.repartition(4).write \
    .mode("overwrite") \
    .parquet(str(repartitioned_path))

for file in repartitioned_path.iterdir():
    print(file.name)

.part-00000-4bddbf73-a966-4ab9-863a-6b680717dd68-c000.snappy.parquet.crc
.part-00001-4bddbf73-a966-4ab9-863a-6b680717dd68-c000.snappy.parquet.crc
.part-00002-4bddbf73-a966-4ab9-863a-6b680717dd68-c000.snappy.parquet.crc
.part-00003-4bddbf73-a966-4ab9-863a-6b680717dd68-c000.snappy.parquet.crc
._SUCCESS.crc
part-00000-4bddbf73-a966-4ab9-863a-6b680717dd68-c000.snappy.parquet
part-00001-4bddbf73-a966-4ab9-863a-6b680717dd68-c000.snappy.parquet
part-00002-4bddbf73-a966-4ab9-863a-6b680717dd68-c000.snappy.parquet
part-00003-4bddbf73-a966-4ab9-863a-6b680717dd68-c000.snappy.parquet
_SUCCESS


`repartition(4)` changes the number of Spark partitions and usually causes shuffle.

## 21. Use `coalesce()` Before Writing

In [ ]:
coalesce_path = base_path / "sales_coalesce_1"

sales_df.coalesce(1).write \
    .mode("overwrite") \
    .parquet(str(coalesce_path))

for file in coalesce_path.iterdir():
    print(file.name)

`coalesce(1)` reduces output to one data part file for small demos. Avoid using it blindly on large data.

## 22. Partitioned Write with Multiple Columns

In [17]:
multi_partition_path = base_path / "sales_by_city_product"

sales_df.write \
    .mode("overwrite") \
    .partitionBy("city", "product") \
    .parquet(str(multi_partition_path))

for city_folder in sorted(multi_partition_path.iterdir()):
    print(city_folder.name)

._SUCCESS.crc
_SUCCESS
city=Delhi
city=Mumbai
city=Pune


This creates nested folder levels:

```text
city=Delhi/product=Laptop/
city=Delhi/product=Mobile/
...
```

## 23. Reading Multiple Parquet Paths

In [19]:
part1_path = base_path / "parquet_part1"
part2_path = base_path / "parquet_part2"

sales_df.filter(sales_df.city == "Delhi").write \
    .mode("overwrite") \
    .parquet(str(part1_path))

sales_df.filter(sales_df.city != "Delhi").write \
    .mode("overwrite") \
    .parquet(str(part2_path))

# combined_df = spark.read.parquet(
#     str(part1_path),
#     str(part2_path)
# )

combined_df = spark.read.parquet(
    str(base_path / "parquet_part*")
)

combined_df.show()

+--------+-----------+-------------+------+-------+--------+-------+
|order_id|customer_id|customer_name|  city|product|quantity| amount|
+--------+-----------+-------------+------+-------+--------+-------+
|     102|          2|        Priya|Mumbai| Mobile|       2|40000.0|
|     104|          4|        Sneha|  Pune| Laptop|       1|60000.0|
|     105|          5|        Karan|Mumbai|Monitor|       2|30000.0|
|     101|          1|         Amit| Delhi| Laptop|       1|55000.0|
|     103|          3|        Rahul| Delhi| Tablet|       1|25000.0|
|     106|          6|        Meera| Delhi| Mobile|       1|22000.0|
+--------+-----------+-------------+------+-------+--------+-------+



## 24. Schema Evolution Example

Schema evolution means the schema changes over time.

Example:

```text
Old files:
order_id, customer_name, city, amount

New files:
order_id, customer_name, city, amount, discount
```

## 25. Create Two Parquet Datasets with Different Schemas

In [20]:
old_schema_path = base_path / "schema_evolution" / "old"
new_schema_path = base_path / "schema_evolution" / "new"

old_df = sales_df.select(
    "order_id",
    "customer_name",
    "city",
    "amount"
)

new_df = sales_df.select(
    "order_id",
    "customer_name",
    "city",
    "amount"
).withColumn(
    "discount",
    sales_df.amount * 0.10
)

old_df.write.mode("overwrite").parquet(str(old_schema_path))
new_df.write.mode("overwrite").parquet(str(new_schema_path))

## 26. Read with Schema Merging

In [21]:
merged_schema_df = spark.read \
    .option("mergeSchema", True) \
    .parquet(
        str(old_schema_path),
        str(new_schema_path)
    )

merged_schema_df.show()
merged_schema_df.printSchema()

+--------+-------------+------+-------+--------+
|order_id|customer_name|  city| amount|discount|
+--------+-------------+------+-------+--------+
|     101|         Amit| Delhi|55000.0|  5500.0|
|     102|        Priya|Mumbai|40000.0|  4000.0|
|     103|        Rahul| Delhi|25000.0|  2500.0|
|     104|        Sneha|  Pune|60000.0|  6000.0|
|     105|        Karan|Mumbai|30000.0|  3000.0|
|     106|        Meera| Delhi|22000.0|  2200.0|
|     101|         Amit| Delhi|55000.0|    NULL|
|     102|        Priya|Mumbai|40000.0|    NULL|
|     103|        Rahul| Delhi|25000.0|    NULL|
|     104|        Sneha|  Pune|60000.0|    NULL|
|     105|        Karan|Mumbai|30000.0|    NULL|
|     106|        Meera| Delhi|22000.0|    NULL|
+--------+-------------+------+-------+--------+

root
 |-- order_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- discount: double (nullable = true)



`mergeSchema=True` can combine schema from different Parquet files. Use it carefully because it can add extra cost.

## 27. Parquet and Nested Data

In [22]:
nested_data = [
    (1, "Amit", {"city": "Delhi", "state": "Delhi"}),
    (2, "Priya", {"city": "Mumbai", "state": "Maharashtra"})
]

nested_schema = "customer_id INT, name STRING, address STRUCT<city:STRING,state:STRING>"

nested_df = spark.createDataFrame(
    nested_data,
    schema=nested_schema
)

nested_df.show(truncate=False)
nested_df.printSchema()

+-----------+-----+---------------------+
|customer_id|name |address              |
+-----------+-----+---------------------+
|1          |Amit |{Delhi, Delhi}       |
|2          |Priya|{Mumbai, Maharashtra}|
+-----------+-----+---------------------+

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)



## 28. Write and Read Nested Parquet

In [24]:
nested_parquet_path = base_path / "nested_parquet"

nested_df.write \
    .mode("overwrite") \
    .parquet(str(nested_parquet_path))

nested_parquet_df = spark.read.parquet(str(nested_parquet_path))

nested_parquet_df.show(truncate=False)
nested_parquet_df.printSchema()

+-----------+-----+---------------------+
|customer_id|name |address              |
+-----------+-----+---------------------+
|1          |Amit |{Delhi, Delhi}       |
|2          |Priya|{Mumbai, Maharashtra}|
+-----------+-----+---------------------+

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- state: string (nullable = true)



## 29. Select Nested Fields from Parquet

In [25]:
nested_parquet_df.select(
    "customer_id",
    "name",
    "address.city",
    "address.state"
).show()

+-----------+-----+------+-----------+
|customer_id| name|  city|      state|
+-----------+-----+------+-----------+
|          1| Amit| Delhi|      Delhi|
|          2|Priya|Mumbai|Maharashtra|
+-----------+-----+------+-----------+



## 30. Parquet vs CSV vs JSON

| Feature | CSV | JSON | Parquet |
|---|---|---|---|
| Human readable | Yes | Yes | No |
| Stores schema | Poor | Partial | Strong |
| Columnar | No | No | Yes |
| Compression | Limited | Limited | Strong |
| Nested data | No | Yes | Yes |
| Best use | Simple exchange | APIs/logs/events | Analytics/data lake |
| Performance | Lower | Lower | Higher |

## 46. Stop SparkSession

In [ ]:
spark.stop()